# FiLM: Frame Interpolation for Large Motion

This notebook is a convenience wrapper around a pre-trained model released by ML researchers at Google. Learn more about their work here:

* https://github.com/google-research/frame-interpolation

---

Notebook authored by David Marx ([@DigThatData](https://twitter.com/DigThatData)), released under the MIT license.

If you experience any issue with or have suggestions regarding this notebook, please report them here:
* https://github.com/pytti-tools/pytti-notebook/issues/new


In [ ]:
!git clone https://github.com/pytti-tools/frame-interpolation

Cloning into 'frame-interpolation'...
remote: Enumerating objects: 179, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 179 (delta 61), reused 46 (delta 46), pack-reused 89 (from 1)
Receiving objects: 100% (179/179), 26.97 MiB | 42.69 MiB/s, done.
Resolving deltas: 100% (84/84), done.


In [ ]:
%pip install -r ./frame-interpolation/requirements_colab.txt
%pip install ./frame-interpolation
%pip install apache-beam
%pip install loguru
%pip install mediapy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 27.8 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Processing ./frame-interpolation
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyttitools-film: filename=pyttitools_film-0.0.1-py3-none-any.whl size=54528 sha256=65cdf5782195c6956a714d0ef28b5fa92065a1ff882385e01291e7f2f41b9e24
  Stored in directory: /root/.cache/pip/wheels/28/17/7e/1919d49c44aadcae99d751de540f043c6600f7f87d63

In [ ]:
# This top bit only neneds to be run once
# ...may as well set up the notebook to
# assume the user is only going to run all
# of setup once.
from pathlib import Path
import os

drive_mounted = False
gdrive_fpath = '.'
local_path = '/content/'

##############################

#@markdown # 1. Setup Workspace

#@markdown Run this cell to perform setup.

#@markdown Mounting your google drive is optional.
#@markdown If you mount your drive, code and models will be downloaded to it.
#@markdown This should reduce setup time after your first run.

###################

# Optionally Mount GDrive

mount_gdrive = True # @param{type:"boolean"}
if mount_gdrive and not drive_mounted:
    from google.colab import drive

    gdrive_mountpoint = '/content/drive/' #@param{type:"string"}
    gdrive_subdirectory = 'MyDrive/Local/art_works_drive/zoetrope/FILM/interpolation' #@param{type:"string"}
    gdrive_fpath = str(Path(gdrive_mountpoint) / gdrive_subdirectory)
    try:
        drive.mount(gdrive_mountpoint, force_remount = True)
        !mkdir -p {gdrive_fpath}
        %cd {gdrive_fpath}
        local_path = gdrive_fpath
        drive_mounted = True
    except OSError:
        print(
            "\n\n-----[PYTTI-TOOLS]-------\n\n"
            "If you received a scary OSError and your drive"
            " was already mounted, ignore it."
            "\n\n-----[PYTTI-TOOLS]-------\n\n"
            )
        raise

#####################

# Perform rest of setup

if not Path('./frame-interpolation').exists():
    !git clone https://github.com/pytti-tools/frame-interpolation

try:
    import frame_interpolation
except ModuleNotFoundError:
    %pip install -r ./frame-interpolation/requirements_colab.txt
    %pip install ./frame-interpolation

#url = "https://drive.google.com/drive/folders/1GhVNBPq20X7eaMsesydQ774CgGcDGkc6?usp=sharing"
share_id = "1GhVNBPq20X7eaMsesydQ774CgGcDGkc6"

if not (Path(local_path) / 'saved_model').exists():
    !pip install --upgrade gdown
    !gdown --folder {share_id}

# create default frame
!mkdir -p frames

Mounted at /content/drive/
/content/drive/MyDrive/Local/art_works_drive/zoetrope/interpolation
Cloning into 'frame-interpolation'...
remote: Enumerating objects: 179, done.
remote: Counting objects: 100% (90/90), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 179 (delta 61), reused 46 (delta 46), pack-reused 89 (from 1)
Receiving objects: 100% (179/179), 26.97 MiB | 23.00 MiB/s, done.
fatal: premature end of pack file, 541 bytes missing
fatal: fetch-pack: invalid index-pack output


In [ ]:
#@markdown # 2. Setup Interpolation

#@markdown Specify the directory containing your video frames with the `frames_dir` parameter.

frames_dir = "frames" #@param{'type':'string'}

#@markdown A single pass of the interpolation procedure adds a frame between each contiguous pair of frames in `frames_dir`.

#@markdown If you start with $n$ frames in `frames_dir` and set `recursive_interpolation_passes` to $k$, your total number of frames
#@markdown after interpolation will be:
#@markdown $$2^k (n-1) -1$$

recursive_interpolation_passes = 1 #@param{'type':'integer'}

#@markdown Check this box to generate a video output. If no output video will be generated, the FPS option can be ignored.
output_video = False #@param{'type':'boolean'}
output_video_fps = 10 #@param{'type':'number'}

In [ ]:
#@markdown # 3. Interpolate!
#@markdown Results will be saved in a subdirectory of `frames_dir` named "interpolated_frames"

from loguru import logger

logger.info("Beginning interpolation...")

if output_video:
  !python -m frame_interpolation.eval.interpolator_cli \
      --model_path ./saved_model \
      --pattern {frames_dir} \
      --fps {output_video_fps} \
      --output_video


else:
    !python -m frame_interpolation.eval.interpolator_cli \
      --model_path ./saved_model \
      --pattern {frames_dir} \

logger.info("Interpolation comlpete.")

2025-07-31 08:37:53.776 | INFO     | __main__:<cell line: 0>:6 - Beginning interpolation...


2025-07-31 08:37:54.206183: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753951074.226166    9316 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753951074.232163    9316 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-07-31 08:37:54.251801: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-07-31 08:37:58.355 | DEBUG    | __main__:_run_pipeline:171 - ['frames']
W0731 08:37:58.393157 140379410980864 core.py:15

2025-07-31 08:39:22.418 | INFO     | __main__:<cell line: 0>:14 - Interpolation comlpete.


```
MIT License

Copyright (c) 2022 David Marx

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
```